# M³ step and tool probe

Runs M³ on 5 PANGAEA datasets and 5 iNaturalist species with `default-text-large`. For each file it records the plan steps, the tools the players call, and the wall-clock time of the run.

Results are written to `outputs/m3_step_probe/`. A successful row that already has `runtime_seconds` is skipped, so the cell can be run again after an interruption. Rows from before timing was recorded are run again.

In [1]:
import logging
import os
import re
import sys
from collections import Counter
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

METADATA_AGENT_ROOT = Path('/home/com3dian/Github/metadata_agent')
REPO = Path('/home/com3dian/Github/process_data_metadata_agent')
sys.path.insert(0, str(METADATA_AGENT_ROOT))

load_dotenv(METADATA_AGENT_ROOT / '.env')
load_dotenv(REPO / '.env', override=True)
os.environ['LLM_MODEL'] = 'default-text-large'
os.environ.setdefault('SURF_API_BASE', 'https://willma.surf.nl/api/v0/')

from src.config import LLM_PROVIDER, get_model_name
from src.orchestrator import Orchestrator
from src.standards import METADATA_STANDARDS

MODEL_NAME = get_model_name('default-text-large')
N_DATASETS = 5
TOPOLOGY = 'single'
SUMMARY_PATH = REPO / 'outputs' / 'm3_step_probe' / 'default-text-large_summary.csv'
SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)

RUNS = [
    (
        'pangaea',
        REPO / 'outputs' / 'pangaea' / 'pangaea_datasets',
        'croissant_pangaea_standard',
    ),
    (
        'inaturalist',
        REPO / 'outputs' / 'inaturalist' / 'inaturalist_100_species_1000_obs',
        'croissant_inaturalist_standard',
    ),
]

print(f'Provider: {LLM_PROVIDER}')
print(f'Model: {MODEL_NAME}')
for corpus, folder, _standard in RUNS:
    files = sorted(folder.glob('*.csv'))[:N_DATASETS]
    print(f'{corpus}: {len(files)} files from {folder}')
    for path in files:
        print(f'  {path.name}')

Provider: surf
Model: default-text-large
pangaea: 5 files from /home/com3dian/Github/process_data_metadata_agent/outputs/pangaea/pangaea_datasets
  897882.csv
  899705.csv
  901488.csv
  901510.csv
  902576.csv
inaturalist: 5 files from /home/com3dian/Github/process_data_metadata_agent/outputs/inaturalist/inaturalist_100_species_1000_obs
  acianthus_fornicatus.csv
  acontia_cretata.csv
  acrothamnus_colensoi.csv
  agromyza_aristata.csv
  alopecurus_myosuroides.csv


In [2]:
import time

ITERATION_FINISHED = re.compile(r'finished after (\d+) iteration')
ITERATION_CAPPED = re.compile(r'hit max iterations \((\d+)\)')


class IterationCounter(logging.Handler):
    def __init__(self):
        super().__init__(level=logging.INFO)
        self.iterations = 0

    def emit(self, record):
        message = record.getMessage()
        finished = ITERATION_FINISHED.search(message)
        capped = ITERATION_CAPPED.search(message)
        if finished:
            self.iterations += int(finished.group(1))
        elif capped:
            self.iterations += int(capped.group(1))


def load_summary():
    if SUMMARY_PATH.exists():
        return pd.read_csv(SUMMARY_PATH)
    return pd.DataFrame()


def already_done(summary, corpus, csv_name):
    if summary.empty or 'runtime_seconds' not in summary.columns:
        return False
    matched = summary[(summary['corpus'] == corpus) & (summary['csv'] == csv_name)]
    if matched.empty:
        return False
    last = matched.iloc[-1]
    return bool(last['success']) and pd.notna(last['runtime_seconds'])


def probe_one(corpus, csv_path, standard_name):
    counter = IterationCounter()
    logging.getLogger().addHandler(counter)
    started = time.perf_counter()
    orchestrator = Orchestrator(topology_name=TOPOLOGY)
    try:
        result = orchestrator.run(
            source=str(csv_path.resolve()),
            metadata_standard=METADATA_STANDARDS[standard_name],
            metadata_standard_name=standard_name,
            name=f'{corpus}_{csv_path.stem}',
        )
    finally:
        elapsed = time.perf_counter() - started
        logging.getLogger().removeHandler(counter)

    tools = orchestrator.executor.list_tools_called() if result is not None else []
    counts = Counter(tools)
    return {
        'corpus': corpus,
        'csv': csv_path.name,
        'model': MODEL_NAME,
        'success': bool(result and result.success),
        'plan_steps': None if result is None else result.plan_steps_count,
        'steps_completed': None if result is None else result.steps_completed,
        'tool_loop_iterations': counter.iterations,
        'tool_calls': len(tools),
        'unique_tools': len(counts),
        'tools': ', '.join(f'{name} x{count}' for name, count in counts.most_common()),
        'runtime_seconds': round(elapsed, 3),
        'error': None if result is None else result.error,
    }


logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')
summary = load_summary()
rows = []
for corpus, folder, standard_name in RUNS:
    for csv_path in sorted(folder.glob('*.csv'))[:N_DATASETS]:
        if already_done(summary, corpus, csv_path.name):
            print(f'skip {corpus} {csv_path.name}')
            continue
        print(f'run {corpus} {csv_path.name}', flush=True)
        started = time.perf_counter()
        try:
            row = probe_one(corpus, csv_path, standard_name)
        except Exception as error:
            row = {
                'corpus': corpus,
                'csv': csv_path.name,
                'model': MODEL_NAME,
                'success': False,
                'plan_steps': None,
                'steps_completed': None,
                'tool_loop_iterations': None,
                'tool_calls': None,
                'unique_tools': None,
                'tools': '',
                'runtime_seconds': round(time.perf_counter() - started, 3),
                'error': str(error),
            }
            print(f'failed {csv_path.name}: {error}')
        rows.append(row)
        if not summary.empty:
            summary = summary[
                ~((summary['corpus'] == corpus) & (summary['csv'] == csv_path.name))
            ]
        summary = pd.concat([summary, pd.DataFrame([row])], ignore_index=True)
        summary.to_csv(SUMMARY_PATH, index=False)
        print(
            f"  steps {row['steps_completed']}/{row['plan_steps']}, "
            f"tool-loop iterations {row['tool_loop_iterations']}, "
            f"tool calls {row['tool_calls']}, "
            f"runtime {row['runtime_seconds']}s"
        )

print(f'Saved {SUMMARY_PATH}')
summary

run pangaea 897882.csv


/tmp/ipykernel_274847/3185349389.py:36: UserWarning: Parameters {'extra_body'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  orchestrator = Orchestrator(topology_name=TOPOLOGY)
INFO root: PlanExecutor initialized with topology: single
INFO root:   Players per step: 1
INFO root:   Debate rounds: 0
INFO root:   Player pool: ['dataset_schema_preview', 'data_analyst', 'schema_expert', 'metadata_specialist', 'relationship_analyst', 'spatial_temporal_specialist', 'critic']
INFO root: Orchestrator initialized with topology: single
INFO root: ============================================================
INFO root: STARTING ORCHESTRATION
INFO root: Context: pangaea_897882
INFO root: Type: single_csv
INFO root: Resources: ['897882']
INFO root: Metadata standard: croissant_pangaea_standard (structured output)
INFO root: ============================================================
INFO root: [ui] planning
INFO root: ==============================

  steps 4/4, tool-loop iterations 11, tool calls 8
run pangaea 899705.csv


/tmp/ipykernel_274847/3185349389.py:36: UserWarning: Parameters {'extra_body'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  orchestrator = Orchestrator(topology_name=TOPOLOGY)
INFO root: PlanExecutor initialized with topology: single
INFO root:   Players per step: 1
INFO root:   Debate rounds: 0
INFO root:   Player pool: ['dataset_schema_preview', 'data_analyst', 'schema_expert', 'metadata_specialist', 'relationship_analyst', 'spatial_temporal_specialist', 'critic']
INFO root: Orchestrator initialized with topology: single
INFO root: ============================================================
INFO root: STARTING ORCHESTRATION
INFO root: Context: pangaea_899705
INFO root: Type: single_csv
INFO root: Resources: ['899705']
INFO root: Metadata standard: croissant_pangaea_standard (structured output)
INFO root: ============================================================
INFO root: [ui] planning
INFO root: ==============================

  steps 4/4, tool-loop iterations 9, tool calls 6
run pangaea 901488.csv


/tmp/ipykernel_274847/3185349389.py:36: UserWarning: Parameters {'extra_body'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  orchestrator = Orchestrator(topology_name=TOPOLOGY)
INFO root: PlanExecutor initialized with topology: single
INFO root:   Players per step: 1
INFO root:   Debate rounds: 0
INFO root:   Player pool: ['dataset_schema_preview', 'data_analyst', 'schema_expert', 'metadata_specialist', 'relationship_analyst', 'spatial_temporal_specialist', 'critic']
INFO root: Orchestrator initialized with topology: single
INFO root: ============================================================
INFO root: STARTING ORCHESTRATION
INFO root: Context: pangaea_901488
INFO root: Type: single_csv
INFO root: Resources: ['901488']
INFO root: Metadata standard: croissant_pangaea_standard (structured output)
INFO root: ============================================================
INFO root: [ui] planning
INFO root: ==============================

... (12 outputs left out)

  steps 2/2, tool-loop iterations 7, tool calls 6
run inaturalist alopecurus_myosuroides.csv


/tmp/ipykernel_274847/3185349389.py:36: UserWarning: Parameters {'extra_body'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  orchestrator = Orchestrator(topology_name=TOPOLOGY)
INFO root: PlanExecutor initialized with topology: single
INFO root:   Players per step: 1
INFO root:   Debate rounds: 0
INFO root:   Player pool: ['dataset_schema_preview', 'data_analyst', 'schema_expert', 'metadata_specialist', 'relationship_analyst', 'spatial_temporal_specialist', 'critic']
INFO root: Orchestrator initialized with topology: single
INFO root: ============================================================
INFO root: STARTING ORCHESTRATION
INFO root: Context: inaturalist_alopecurus_myosuroides
INFO root: Type: single_csv
INFO root: Resources: ['alopecurus_myosuroides']
INFO root: Metadata standard: croissant_inaturalist_standard (structured output)
INFO root: ============================================================
INFO root: [ui] planning
I

  steps 2/2, tool-loop iterations 7, tool calls 6
Saved /home/com3dian/Github/process_data_metadata_agent/outputs/m3_step_probe/default-text-large_summary.csv


,corpus,csv,model,success,plan_steps,steps_completed,tool_loop_iterations,tool_calls,unique_tools,tools,error
0,pangaea,897882.csv,default-text-large,True,4,4,11,8,7,"get_unique_values x2, get_columns_with_first_r...",None
1,pangaea,899705.csv,default-text-large,True,4,4,9,6,5,"get_unique_values x2, get_columns_with_first_r...",None
2,pangaea,901488.csv,default-text-large,True,4,4,11,8,7,"get_unique_values x2, get_columns_with_first_r...",None
3,pangaea,901510.csv,default-text-large,True,4,4,11,8,7,"get_unique_values x2, get_columns_with_first_r...",None
4,pangaea,902576.csv,default-text-large,True,4,4,10,7,5,"get_unique_values x3, get_columns_with_first_r...",None
5,inaturalist,acianthus_fornicatus.csv,default-text-large,True,2,2,8,7,4,"get_temporal_extent x3, detect_spatial_columns...",None
6,inaturalist,acontia_cretata.csv,default-text-large,True,2,2,7,6,4,"get_temporal_extent x3, detect_temporal_column...",None
7,inaturalist,acrothamnus_colensoi.csv,default-text-large,True,2,2,7,6,4,"get_temporal_extent x3, detect_temporal_column...",None
8,inaturalist,agromyza_aristata.csv,default-text-large,True,2,2,7,6,4,"get_temporal_extent x3, detect_temporal_column...",None
9,inaturalist,alopecurus_myosuroides.csv,default-text-large,True,2,2,7,6,4,"get_temporal_extent x3, detect_temporal_column...",None
